## Building a nano GPT II - using bigrams and frequent trigrams

Companion notebook to the [Zero To Hero](https://karpathy.ai/zero-to-hero.html) video on GPT.

In [ ]:
# We always start with a dataset to train on. Let's download the tiny shakespeare dataset
# !wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
remove_new_lines = True
to_lower = True
add_bigrams = False
add_trigrams = False

In [ ]:
# read it in to inspect it
# fName = 'input.txt' # shakespeare
# fName = 'input_sherlock.txt'
fName = 'input_sherlock_3.txt'

# fName = 'input_preseren.txt' # vsa dela
# fName = 'input_preseren_2.txt' # krst pri savici
# fName = 'input_preseren_3.txt' # sonetni venec


with open(fName, 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
if remove_new_lines:
    text = text.replace("\n", " ")
text = text.replace(",", ", ")
text = text.replace(".", ". ")
text = text.replace(";", "; ")
if to_lower:
    text = text.lower()
text = text.replace("  ", " ")
text = text.replace("  ", " ")
text = text.replace("  ", " ")

In [ ]:
print("length of dataset in characters: ", len(text))

In [ ]:
# let's look at the first 1000 characters
print(text[:3000])

In [ ]:
# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
# mejni_index = 10 # input_preseren_3.txt
# mejni_index = 17 # input_preseren_2.txt
mejni_index = 13 # input_sherlock_3.txt
locila = chars[:mejni_index]
letters = chars[mejni_index:]

bidict = {}
for i in range(len(text)-1):
    if (text[i] in letters):
        if (text[i+1] in letters):
            if text[i:i+2] in bidict:
                bidict[text[i:i+2]] += 1
            else:
                bidict[text[i:i+2]] = 1

print('Bigrami min, avg, max: ', min(bidict.values()), sum(bidict.values())/len(bidict), max(bidict.values()))
avg = sum(bidict.values())/len(bidict)
bigrams = [k for (k, v) in bidict.items()]
if add_bigrams:
    chars.extend(bigrams)

tridict = {}
for i in range(len(text)-2):
    if (text[i] in letters):
        if (text[i+1] in letters):
            if (text[i+2] in letters):
                if text[i:i+3] in tridict:
                    tridict[text[i:i+3]] += 1
                else:
                    tridict[text[i:i+3]] = 1

print('Trigrami min, avg, max: ', min(tridict.values()), sum(tridict.values())/len(tridict), max(tridict.values()))
avg = sum(tridict.values())/len(tridict)
trigrams = [k for (k, v) in tridict.items() if v > avg]
if add_trigrams:
    chars.extend(trigrams)

vocab_size = len(chars)
print('Slovar: ', len(chars), chars)
print('Ločila: ', len(locila), locila)
print('Črke: ', len(letters), letters)
print('Bigrami: ', len(bigrams), bigrams[:50])
print(len(bidict), bidict)
print('Trigrami: ', len(trigrams), trigrams[:100])
print(len(tridict), tridict)
print('|'.join(chars))
print(vocab_size)

In [ ]:
# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode_old = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

def encode(s):
    # encoder: take a string, output a list of integers
    encoded = []
    i = 0
    while (i < len(s)):
        if (i == len(s)-1): # on the last character in s
            encoded.append(stoi[s[i]])
            i += 1
        elif (i == len(s)-2): # on the last two characters in s
            ch = s[i:i+2]
            if ch in chars:
                encoded.append(stoi[ch])
                i += 2
            else:
                encoded.append(stoi[s[i]])
                i += 1
        else: # before the last two characters in s
            ch = s[i:i+3]
            if ch in chars:
                encoded.append(stoi[ch])
                i += 3
            else:
                ch = s[i:i+2]
                if ch in chars:
                    encoded.append(stoi[ch])
                    i += 2
                else:
                    encoded.append(stoi[s[i]])
                    i += 1
    return encoded

test_str = '''poet tvoj nov slovencam venec vije, 
'z petnajst sonetov ti tako ga spleta, 
de "magistrale", pesem trikrat peta, 
vseh drugih skupej veže harmonije.'''

test_str = '''gnijo po polji v bojih pokončani
trum srčni vajvodi, in njih vojšaki,
sam črtomir se z majhnim tropam brani.'''

test_str = '''to sherlock holmes she is always the woman. i have seldom heard him mention her under any other name. in his eyes she eclipses and predominates the whole of her sex.'''

print(encode(test_str))
print(decode(encode(test_str)))
for i in encode(test_str):
    print(i, ': #', itos[i], '#', sep='')

In [ ]:
def tensor_to_string(tensor):
    # Check if the tensor is 1-dimensional
    if tensor.dim() != 1:
        raise ValueError("Input tensor must be 1-dimensional")
    
    string = decode(tensor.tolist())
    return string

import torch
data_test = torch.tensor(encode(text), dtype=torch.long)
tensor_to_string(data_test[:100])

In [ ]:
# let's now encode the entire text dataset and store it into a torch.Tensor
import torch # we use PyTorch: https://pytorch.org
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000]) # the 1000 characters we looked at earier will to the GPT look like this

print(tensor_to_string(data[:1000]))

In [ ]:
print(len(data), data[:100])

In [ ]:
# Let's now split up the data into train and validation sets
n = int(0.95*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [ ]:
block_size = 8
train_data[:block_size+1]

In [ ]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} the target: {target}")

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # how many independent sequences will we process in parallel?
block_size = 8 # what is the maximum context length for predictions?

def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size): # batch dimension
    for t in range(block_size): # time dimension
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
print(xb) # our input to the transformer

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):

        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C)
        
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # get the predictions
            logits, loss = self(idx)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))


In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(100): # increase number of steps for good results... 
    
    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())


In [ ]:
print(decode(m.generate(idx = torch.zeros((1, 1), dtype=torch.long), max_new_tokens=300)[0].tolist()))

## The mathematical trick in self-attention

In [ ]:
# toy example illustrating how matrix multiplication can be used for a "weighted aggregation"
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

In [ ]:
# consider the following toy example:

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

In [ ]:
# We want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)


In [ ]:
# version 2: using matrix multiply for a weighted aggregation
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

In [ ]:
# version 3: use Softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)


In [ ]:
# version 4: self-attention!
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

In [ ]:
wei[0]

Notes:
- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.
- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.
- Each example across batch dimension is of course processed completely independently and never "talk" to each other
- In an "encoder" attention block just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.
- "self-attention" just means that the keys and values are produced from the same source as queries. In "cross-attention", the queries still get produced from x, but the keys and values come from some other, external source (e.g. an encoder module)
- "Scaled" attention additional divides `wei` by 1/sqrt(head_size). This makes it so when input Q,K are unit variance, wei will be unit variance too and Softmax will stay diffuse and not saturate too much. Illustration below

In [ ]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [ ]:
k.var()

In [ ]:
q.var()

In [ ]:
wei.var()

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

In [ ]:
class LayerNorm1d: # (used to be BatchNorm1d)
  
  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)
  
  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out
  
  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

In [ ]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

In [ ]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

In [ ]:
# place for testing
ix = torch.randint(len(data) - block_size, (batch_size,))
ix

### Full finished code, for reference

#### Estimation of model parameters

* n_embd = 64 (embedding vector size)
* n_head = 4 ⇒ head_size = 16 (n_embd // n_head)
* n_layer = 4 (4 Transformer blocks stacked on top of each other: Block1 → Block2 → Block3 → Block4. <br>
Each Block is a Transformer layer, refining the token representations computed by the previous layer, consisting of:
    - Multi-Head Self-Attention (MHA) – models contextual relationships between tokens.
    - Feed-Forward Network (FFN) – processes the output of attention with nonlinear transformations.
    - Layer Normalization and residual connections for stability and gradient flow.
* block_size = 64 (the maximum context length for predictions)
* vocabulary size (e.g. 42)

##### 1. Embeddings

* Token embedding nn.Embedding(V, 64), Params: V × 64
* Position embedding nn.Embedding(64, 64), Params: 64 × 64 = 4,096

##### 2. Each Transformer block (there are 4)

* Multi-Head Self-Attention
    - Per head (64 → 16, no bias):
        - key: 64×16 = 1,024
        - query: 64×16 = 1,024
        - value: 64×16 = 1,024
        - Per head total: 3,072
        - 4 heads ⇒ 12,288

    - Output projection Linear(64 → 64, bias=True):
        - weights: 64×64 = 4,096
        - bias: 64
        - Output proj total: 4,160

    - MHA total per block: 12,288 + 4,160 = 16,448

* Feed-Forward (MLP)
    - Linear(64 → 256, bias=True):
        - weights: 64×256 = 16,384
        - bias: 256
        - Subtotal: 16,640

    - Linear(256 → 64, bias=True):
        - weights: 256×64 = 16,384
        - bias: 64
        - Subtotal: 16,448

    - FFN total per block: 16,640 + 16,448 = 33,088

* LayerNorms (affine=True)
    - Two LNs per block, each has weight(64) + bias(64) = 128
    - LN total per block: 2 × 128 = 256

* Grand total per Transformer block:
    - MHA (16,448) + FFN (33,088) + LNs (256) = 49,792

* 4 Transformer blocks ⇒ 4 × 49,792 = 199,168

##### 3. Final LayerNorm

* LN(64) ⇒ weight(64) + bias(64) = 128

##### 4. Output head

* Linear(64 → V, bias=True)
    - weights: 64 × V
    - bias: V
    - Total: 64 × V + V = 65 × V

#### Grand totals

* Add everything up:
    - Embeddings: V × 64 + 4,096
    - 4 Blocks: 199,168
    - Final LN: 128
    - LM Head: 65 × V

* Total parameters = 203,392 + 129 × V
    - (where 203,392 = 4,096 + 199,168 + 128)

* For V = 42: 208,810 parameters (0.20881 M parameters)

* Comparison: GPT-3: 175 B parameters (roughly a Million times more)

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32 # 128 # 64 # 32 # 16 # how many independent sequences will we process in parallel?
block_size = 64 # 128 # 64 # 32 # 16 # what is the maximum context length for predictions?
max_iters = 3000
eval_interval = 200
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

eval_iters = 200
n_embd = 64 # 128 # 64 # 32 # 16 #        ### GPT-3: 12.288, vocabulary size 50.257
n_head = 4                                ### GPT-3: 96
n_layer = 4                               ### GPT-3: 96

dropout = 0.0
keep_only_the_best = False
# head_size = n_embd // n_head = 64 / 4 = 16
# ------------

torch.manual_seed(1337)

# wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
# with open('input.txt', 'r', encoding='utf-8') as f:
with open(fName, 'r', encoding='utf-8') as f:
    text = f.read()

if remove_new_lines:
    text = text.replace("\n", " ")
text = text.replace(",", ", ")
text = text.replace(".", ". ")
text = text.replace(";", "; ")
if to_lower:
    text = text.lower()
text = text.replace("  ", " ")
text = text.replace("  ", " ")
text = text.replace("  ", " ")

# here are all the unique characters that occur in this text
chars = sorted(list(set(text)))
locila = chars[:mejni_index]
letters = chars[mejni_index:]

bidict = {}
for i in range(len(text)-1):
    if (text[i] in letters):
        if (text[i+1] in letters):
            if text[i:i+2] in bidict:
                bidict[text[i:i+2]] += 1
            else:
                bidict[text[i:i+2]] = 1

bigrams = [k for (k, v) in bidict.items()]
if add_bigrams:
    chars.extend(bigrams)

tridict = {}
for i in range(len(text)-2):
    if (text[i] in letters):
        if (text[i+1] in letters):
            if (text[i+2] in letters):
                if text[i:i+3] in tridict:
                    tridict[text[i:i+3]] += 1
                else:
                    tridict[text[i:i+3]] = 1

avg = sum(tridict.values())/len(tridict)
trigrams = [k for (k, v) in tridict.items() if v > avg]
if add_trigrams:
    chars.extend(trigrams)

vocab_size = len(chars)

# create a mapping from characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode_old = lambda s: [stoi[c] for c in s] # encoder: take a string, output a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decoder: take a list of integers, output a string

def encode(s):
    # encoder: take a string, output a list of integers
    encoded = []
    i = 0
    while (i < len(s)):
        if (i == len(s)-1): # on the last character in s
            encoded.append(stoi[s[i]])
            i += 1
        elif (i == len(s)-2): # on the last two characters in s
            ch = s[i:i+2]
            if ch in chars:
                encoded.append(stoi[ch])
                i += 2
            else:
                encoded.append(stoi[s[i]])
                i += 1
        else: # before the last two characters in s
            ch = s[i:i+3]
            if ch in chars:
                encoded.append(stoi[ch])
                i += 3
            else:
                ch = s[i:i+2]
                if ch in chars:
                    encoded.append(stoi[ch])
                    i += 2
                else:
                    encoded.append(stoi[s[i]])
                    i += 1
    return encoded

# Train and test splits
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.95*len(data)) # first 95% will be train, rest validation
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    # generate a small batch of data of inputs x and targets y
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)   # (B,T,C-head_size)
        q = self.query(x) # (B,T,C)
        # compute attention scores ("affinities")
        wei = q @ k.transpose(-2,-1) * C**-0.5 # (B, T, C) @ (B, C, T) -> (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf')) # (B, T, T)
        wei = F.softmax(wei, dim=-1) # (B, T, T)
        wei = self.dropout(wei)
        # perform the weighted aggregation of the values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B, T, T) @ (B, T, C) -> (B, T, C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedFoward(nn.Module):
    """ a simple linear layer followed by a non-linearity """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        # n_embd: embedding dimension, n_head: the number of heads we'd like
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedFoward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# super simple bigram model
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()
        # each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd) # final layer norm
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # idx and targets are both (B,T) tensor of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x) # (B,T,C)
        x = self.ln_f(x) # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # crop idx to the last block_size tokens
            idx_cond = idx[:, -block_size:]
            # print('idx_shape: ', idx.shape, idx_cond.shape, idx_cond.tolist())
            # get the predictions
            logits, loss = self(idx_cond)
            # focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # print('probs: ', probs)
            # sample from the distribution
            idx_next_1 = torch.multinomial(probs, num_samples=1) # (B, 1)
            _, idx_next_2 = probs.max(dim=1)
            #if keep_only_the_best:
            #    idx_next = torch.tensor([[idx_next_2]])
            #else:
            #    idx_next = idx_next_1
            idx_next = idx_next_2.unsqueeze(1) if keep_only_the_best else idx_next_1
            # print('idx, idx_next: ', idx_cond, idx_next_1, idx_next_2, idx_next)
            # append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

model = BigramLanguageModel()
m = model.to(device)
# print the number of parameters in the model
print(sum(p.numel() for p in m.parameters())/1e6, 'M parameters') ### GPT-3: 175 B parameters

In [ ]:
# before the optimizer with just random settings
# generate from the model
# prompt = 'ale'
prompt = 'holmes'
# prompt = 'king '
context_ids = encode(prompt)
context_ids.insert(0, 0)

context = torch.tensor([context_ids])
print(decode(m.generate(context, max_new_tokens=1000)[0].tolist()))

In [ ]:
# create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    # every once in a while evaluate the loss on train and val sets
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print('')
        print('---------')
        # print(f"step {iter}/{max_iters}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        print(f"step {iter}/{max_iters}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

        # include small generated text
        # prompt = 'ale'
        prompt = 'holmes'
        # prompt = 'king '
        context_ids = encode(prompt)
        context_ids.insert(0, 0)

        context = torch.tensor([context_ids])
        print(decode(m.generate(context, max_new_tokens=250)[0].tolist()))

        # mystr = decode(m.generate(context, max_new_tokens=250)[0].tolist())
        # print(mystr.replace(".", ".\n"))
        

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()



In [ ]:
# inspect various variables
print(model.token_embedding_table)    # token from the vocabulary
print(model.position_embedding_table) # position in the sentence

In [ ]:
print(model.token_embedding_table.weight[1])

In [ ]:
print(model.position_embedding_table.weight[1])

In [ ]:
for i in range(len(itos)):
    print(i, itos[i], model.token_embedding_table.weight[i])
    print(i, itos[i])

In [ ]:
for i in range(block_size):
    print(i, model.position_embedding_table.weight[i])

In [ ]:
myitos = itos.copy()
myitos[0] = '<br>'
myitos[1] = '<sp>'
list(myitos.values())[:10]

In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# ---- Load your model (example: GPT-like model) ----
# Make sure you have your model already defined and loaded.
# Example: model = MyGPTModel()

# Extract embeddings
token_emb = model.token_embedding_table.weight.detach().cpu()
pos_emb = model.position_embedding_table.weight.detach().cpu()

# ---- Choose which embeddings to visualize ----
# For large vocabularies, take a small subset for clarity
n_tokens = 40        # number of tokens to visualize
n_positions = block_size     # number of positions to visualize
token_subset = token_emb[:n_tokens]
pos_subset = pos_emb[:n_positions]

# ---- Reduce dimensions to 2D ----
# Option 1: PCA (fast)
# reducer = PCA(n_components=2)

# Option 2: t-SNE (better for nonlinear patterns)
reducer = TSNE(n_components=2, perplexity=30, random_state=42)

token_2d = reducer.fit_transform(token_subset)
pos_2d = reducer.fit_transform(pos_subset)

# ---- Plot ----
plt.figure(figsize=(10, 6))
plt.scatter(token_2d[:, 0], token_2d[:, 1], c='blue', alpha=0.6, label='Token Embeddings')
# Add a text label near each point
for i in range(min(n_tokens, len(myitos))):
    plt.text(token_2d[i, 0], token_2d[i, 1], myitos[i], fontsize=9)

#plt.scatter(pos_2d[:, 0], pos_2d[:, 1], c='red', alpha=0.6, label='Position Embeddings')
#for i in range(n_positions):
#    plt.text(pos_2d[i, 0], pos_2d[i, 1], str(i), fontsize=9)

plt.title("2D Visualization of Token vs Position Embeddings")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# ---- Load or define your model ----
# Example: model = MyGPTModel()
# (Make sure your model has .token_embedding_table and .position_embedding_table)

# ---- Extract embeddings ----
token_emb = model.token_embedding_table.weight.detach().cpu()
pos_emb = model.position_embedding_table.weight.detach().cpu()

# ---- Choose subsets for visualization ----
n_tokens = min(600, token_emb.shape[0])       # number of tokens to visualize
n_positions = min(600, pos_emb.shape[0])      # number of positions to visualize
token_subset = token_emb[:n_tokens]
pos_subset = pos_emb[:n_positions]

# ---- Optional: Use PCA first for speed (good before t-SNE) ----
pca = PCA(n_components=35) # 50
token_reduced = pca.fit_transform(token_subset)
pos_reduced = pca.fit_transform(pos_subset)

# ---- Reduce to 2D using t-SNE ----
reducer = TSNE(n_components=2, perplexity=30, random_state=42)
token_2d = reducer.fit_transform(token_reduced)
pos_2d = reducer.fit_transform(pos_reduced)

# ---- Plot both embeddings ----
plt.figure(figsize=(12, 8))
plt.scatter(token_2d[:, 0], token_2d[:, 1], c='blue', alpha=0.4, label='Token Embeddings')
# plt.scatter(pos_2d[:, 0], pos_2d[:, 1], c='red', alpha=0.4, label='Position Embeddings')

# ---- Annotate top 50 token embeddings ----
n_labels = min(600, n_tokens)

# If your model has a vocabulary object or tokenizer, map IDs to tokens
# Example: vocab or tokenizer.decode([i])
try:
    # vocab_labels = [model.tokenizer.decode([i]) for i in range(n_labels)]
    vocab_labels = [myitos[i]+':'+str(i) for i in range(n_labels)]

except AttributeError:
    # Fallback: use token IDs if no tokenizer available
    vocab_labels = [f"tok_{i}" for i in range(n_labels)]
print(vocab_labels)

for i, label in enumerate(vocab_labels):
    x, y = token_2d[i, 0], token_2d[i, 1]
    plt.text(x, y, label, fontsize=6, alpha=0.8)

# ---- Finishing touches ----
plt.title("2D Visualization of Token Embeddings")
plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
"""
Hierarchical clustering of token embeddings with a labeled dendrogram.

Requirements:
  pip install scipy matplotlib torch (and your model/tokenizer deps)

Assumptions:
  - `model.token_embedding_table.weight` exists (shape: [vocab_size, embed_dim])
  - (Optional) `model.tokenizer` or `decode` method to map ids -> string tokens

Tips:
  - For readability, keep n_tokens <= ~300.
  - Cosine distance + average linkage is a solid default for embeddings.
"""

import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

# -----------------------------
# 1) Get token embeddings
# -----------------------------
# Define or load your model before running this (must expose token_embedding_table)
# Example: model = MyGPTLikeModel.load_from_checkpoint(...)

# --- extract embeddings ---
token_emb = model.token_embedding_table.weight.detach().cpu().numpy()  # [V, D]
vocab_size = token_emb.shape[0]

# -----------------------------
# 2) Choose which tokens to cluster
# -----------------------------
# Option A: first N tokens
N = min(400, vocab_size)              # adjust as you like (<= ~300 for readable tree)
token_ids = list(range(N))

# Option B (example): pick a custom subset
# token_ids = [0, 1, 2, 3, 4, 5, 198, 199]

X = token_emb[token_ids]              # [N, D]

# -----------------------------
# 3) Build labels for dendrogram
# -----------------------------
def id_to_label(i):
    # Try to decode token id -> string; fall back to "tok_<id>"
    try:
        # Many tokenizers decode a single id via decode([id])
        # s = model.tokenizer.decode([i])
        s = myitos[i]
        # Make short & printable
        s = s.replace("\n", "\\n")
        if len(s) > 12:
            s = s[:9] + "…"
        return s if s.strip() != "" else f"tok_{i}"
    except Exception:
        return f"tok_{i}"

labels = [id_to_label(i) for i in token_ids]

# -----------------------------
# 4) Normalize & distance
# -----------------------------
# Cosine distance benefits from L2-normalization
X_norm = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

# Pairwise distances (cosine)
dists = pdist(X_norm, metric="cosine")  # 1 - cosine_similarity

# -----------------------------
# 5) Hierarchical clustering
# -----------------------------
# Use average linkage for cosine distances
Z = linkage(dists, method="average")  # alternatives: "complete", "single", "weighted"

# -----------------------------
# 6) Plot dendrogram
# -----------------------------
plt.figure(figsize=(14, 7))
# You can truncate the tree if N is large to keep it readable (uncomment next line)
# dn = dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=8, truncate_mode="lastp", p=50)
dn = dendrogram(
    Z,
    labels=labels,
    leaf_rotation=90,
    leaf_font_size=8,
    distance_sort="descending",
    show_contracted=False,
)
plt.title("Hierarchical Clustering of Token Embeddings (cosine, average linkage)")
plt.ylabel("Cosine distance")
plt.tight_layout()
plt.show()

# -----------------------------
# 7) (Optional) Cut the tree into K clusters and print membership
# -----------------------------
K = 3  # choose number of clusters to print; or use distance threshold
clusters = fcluster(Z, t=K, criterion="maxclust")

# Print a compact summary
print(f"\nCluster memberships (K={K}):")
cluster_map = {}
map_cluster = {}
for lbl, cid in zip(labels, clusters):
    cluster_map.setdefault(cid, []).append('"'+lbl+'"')
    map_cluster[lbl] = cid

for cid in sorted(cluster_map):
    print(f"Cluster {cid}: {', '.join(cluster_map[cid])}")

print(map_cluster)

# -----------------------------
# 8) (Optional) Distance heatmap for a subset
# -----------------------------
# If you want a quick look at pairwise distances for the first M tokens
# (Large M makes the heatmap dense.)
M = min(60, N)
Dmat = squareform(pdist(X_norm[:M], metric="cosine"))
plt.figure(figsize=(6, 5))
plt.imshow(Dmat, interpolation="nearest")
plt.colorbar(label="Cosine distance")
plt.xticks(range(M), [labels[i] for i in range(M)], rotation=90, fontsize=7)
plt.yticks(range(M), [labels[i] for i in range(M)], fontsize=7)
plt.title("Pairwise Cosine Distance (subset)")
plt.tight_layout()
plt.show()


In [ ]:
from typing import List, Optional
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from scipy.sparse import csr_matrix
from sklearn.base import BaseEstimator
import seaborn as sns

import plotly.graph_objects as go
from sklearn.decomposition import PCA
import random

In [ ]:
def visualize_embeddings_pca_interactive(names, selected_names, domains_list, tfidf_matrix, transpose = False, color_schema = 0):
    tfidf_matrix_transposed = np.squeeze(np.asarray(tfidf_matrix))
    if transpose:
        tfidf_matrix_transposed = tfidf_matrix_transposed.T

    # Apply PCA
    pca = PCA(n_components=2)
    pca_result = pca.fit_transform(tfidf_matrix_transposed)
 
    # Generate colors for each point
    if color_schema == 0:
        colors = ['rgba(242, 69, 69, 0.2)', 'rgba(245, 233, 67, 0.2)', 'rgba(78, 222, 232, 0.2)', 'rgba(145, 232, 78, 0.2)',
                  'rgba(229, 2, 199, 0.2)', 'rgba(229, 145, 2, 0.2)', 'rgba(2, 229, 32, 0.2)', 'rgba(2, 85, 229, 0.2)']
        colors_centroid = ['rgba(242, 69, 69, 0.9)', 'rgba(245, 233, 67, 0.9)', 'rgba(78, 222, 232, 0.9)', 'rgba(145, 232, 78, 0.9)',
                    'rgba(229, 2, 199, 0.9)', 'rgba(229, 145, 2, 0.9)', 'rgba(2, 229, 32, 0.9)', 'rgba(2, 85, 229, 0.9)']
    elif color_schema == 1:
        colors = ['rgba(229, 2, 199, 0.2)', 'rgba(229, 145, 2, 0.2)', 'rgba(2, 229, 32, 0.2)', 'rgba(2, 85, 229, 0.2)',
                  'rgba(242, 69, 69, 0.2)', 'rgba(245, 233, 67, 0.2)', 'rgba(78, 222, 232, 0.2)', 'rgba(145, 232, 78, 0.2)']
        colors_centroid = ['rgba(229, 2, 199, 0.9)', 'rgba(229, 145, 2, 0.9)', 'rgba(2, 229, 32, 0.9)', 'rgba(2, 85, 229, 0.9)',
                    'rgba(242, 69, 69, 0.9)', 'rgba(245, 233, 67, 0.9)', 'rgba(78, 222, 232, 0.9)', 'rgba(145, 232, 78, 0.9)']
    elif color_schema == 2:
        colors = ['rgba(2, 229, 32, 0.2)', 'rgba(2, 85, 229, 0.2)', 'rgba(229, 2, 199, 0.2)', 'rgba(229, 145, 2, 0.2)', 
                  'rgba(242, 69, 69, 0.2)', 'rgba(245, 233, 67, 0.2)', 'rgba(78, 222, 232, 0.2)', 'rgba(145, 232, 78, 0.2)']
        colors_centroid = ['rgba(2, 229, 32, 0.9)', 'rgba(2, 85, 229, 0.9)', 'rgba(229, 2, 199, 0.9)', 'rgba(229, 145, 2, 0.9)', 
                    'rgba(242, 69, 69, 0.9)', 'rgba(245, 233, 67, 0.9)', 'rgba(78, 222, 232, 0.9)', 'rgba(145, 232, 78, 0.9)']
    elif color_schema == 11:
        colors = ['rgba(255, 0, 0, 0.2)', 'rgba(0, 0, 255, 0.2)']
        colors_centroid = ['rgba(255, 0, 0, 0.9)', 'rgba(0, 0, 255, 0.9)']
    elif color_schema == 12:
        colors = ['rgba(255, 153, 153, 0.2)', 'rgba(153, 153, 255, 0.2)']
        colors_centroid = ['rgba(255, 153, 153, 0.9)', 'rgba(153, 153, 255, 0.9)']
    elif color_schema == 13:
        colors = ['rgba(255, 0, 0, 0.2)', 'rgba(255, 128, 0, 0.2)', 'rgba(0, 0, 255, 0.2)', 'rgba(128, 0, 255, 0.2)']
        colors_centroid = ['rgba(255, 0, 0, 0.9)', 'rgba(255, 128, 0, 0.9)', 'rgba(0, 0, 255, 0.9)', 'rgba(128, 0, 255, 0.9)']
    else:
        colors = ['red', 'green', 'blue', 'yellow', 'black', 'grey', 'violet', 'brown', 'lime', 'cyan']

    # Determine unique clusters
    unique_clusters = list(set(domains_list))
    unique_clusters.sort()
    
    # Compute the centroid of the PCA result
    centroid = pca_result.mean(axis=0)
    
    # Create interactive plot
    fig = go.Figure()

    # PCA Scatter plot with random colors
    for cluster_num in range(len(unique_clusters)):
        cluster_docs_indices = [i for i, label in enumerate(domains_list) if label == unique_clusters[cluster_num]]

        # Compute centroid for the current cluster
        centroid_x = np.mean(pca_result[cluster_docs_indices, 0])
        centroid_y = np.mean(pca_result[cluster_docs_indices, 1])

        fig.add_trace(go.Scatter(x=pca_result[cluster_docs_indices, 0], y=pca_result[cluster_docs_indices, 1], 
                                 mode='markers+text',
                                 # marker=dict(size=8, color=colors[cluster_num]), # , color=colors[cluster_num]
                                 marker=dict(size=8, color=colors[cluster_num], symbol='circle', line=dict(color='rgba(0, 0, 0, 0.1)', width=1)), 
                                 name=unique_clusters[cluster_num],
                                 hovertext=[names[i] for i in cluster_docs_indices], # this text is shown on hover
                                 text='', # [names[i] for i in cluster_docs_indices], # this text is set to show always
                                 textposition='bottom center'))
        
        # Plot the centroid of the current cluster
        fig.add_trace(go.Scatter(x=[centroid_x], y=[centroid_y],
                                 mode='markers+text',
                                 marker=dict(size=16, color=colors_centroid[cluster_num], symbol='star', line=dict(color='rgba(0, 0, 0, 0.5)', width=2)),
                                 name='Centroid ' + unique_clusters[cluster_num],
                                 hovertext='Centroid of ' + unique_clusters[cluster_num],
                                 text=unique_clusters[cluster_num],
                                 textposition='bottom center'))

    if selected_names == []:
        special_cluster_docs_indices = []
    else:
        special_cluster_docs_indices = [i for i, label in enumerate(names) if label in selected_names]

    if selected_names != []:
        fig.add_trace(go.Scatter(x=pca_result[special_cluster_docs_indices, 0], y=pca_result[special_cluster_docs_indices, 1], 
                                    mode='markers+text',
                                    marker=dict(size=10, color='green', symbol='circle', line=dict(color='rgba(0, 0, 0, 1.0)', width=1)), 
                                    name='selected',
                                    hovertext=[names[i] for i in special_cluster_docs_indices], # this text is shown on hover
                                    text='', 
                                    textposition='bottom center'))

    # Plot the centroid of the whole set of documents
    fig.add_trace(go.Scatter(x=[centroid[0]], y=[centroid[1]],
                             mode='markers',
                             marker=dict(size=20, color='black', symbol='cross'),
                             name='The main centroid',
                             hovertext=['The main centroid']))

    fig.update_layout(title="PCA Visualization of TF-IDF Vectors",
                      hovermode='closest',
                      showlegend=True,
                      width=1100,  # Set the width of the figure
                      height=1100)  # Set the height of the figure
    
    fig.show()

In [ ]:
# myitos

In [ ]:
# token_emb = model.token_embedding_table.weight.detach().cpu()
token_emb = model.token_embedding_table.weight.detach().cpu().numpy()  # [V, D]
domains_list = [str(map_cluster[v]) for (l,v) in myitos.items()]
visualize_embeddings_pca_interactive(myitos, [], domains_list, token_emb, color_schema = 0)

In [ ]:
import torch

# ---- Inputs you already have ----
# model.token_embedding_table.weight : [vocab_size, d]
# model.position_embedding_table.weight : [max_pos, d]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tok_w = model.token_embedding_table.weight.detach().to(device)   # [V, D]
pos_w = model.position_embedding_table.weight.detach().to(device) # [P, D]

# ---- Utilities ----
def l2_normalize(x, dim=-1, eps=1e-12):
    return x / (x.norm(dim=dim, keepdim=True) + eps)

def cosine_topk(matrix, query, k=10, exclude_idx=None):
    """
    matrix: [N, D]
    query:  [D] or [1, D]
    returns (topk_scores, topk_indices)
    """
    if query.dim() == 1:
        query = query.unsqueeze(0)  # [1, D]
    # Normalize for cosine similarity
    A = l2_normalize(matrix, dim=1)      # [N, D]
    q = l2_normalize(query, dim=1)       # [1, D]
    sims = (A @ q.t()).squeeze(-1)       # [N]
    if exclude_idx is not None:
        sims[exclude_idx] = -1e9         # exclude exact self
    topk_vals, topk_idx = torch.topk(sims, k=k, largest=True, sorted=True)
    return topk_vals, topk_idx

def id_to_token(i):
    # Try to decode token id -> readable string; fallback to id
    try:
        s = myitos[i] # model.tokenizer.decode([int(i)])
        s = s.replace("\n", "\\n")
        if len(s) > 20:
            s = s[:17] + "…"
        return s if s.strip() else f"tok_{i}"
    except Exception:
        return f"tok_{i}"

# ---- 1) Neighbors of a token id among TOKENS and POSITIONS ----
def nearest_to_token(token_id, k_tokens=10, k_positions=10):
    q = tok_w[token_id]  # [D]
    tok_vals, tok_idx = cosine_topk(tok_w, q, k=k_tokens, exclude_idx=token_id)
    pos_vals, pos_idx = cosine_topk(pos_w, q, k=k_positions, exclude_idx=None)

    print(f"Closest tokens to token_id={token_id} ({id_to_token(token_id)}):")
    for r, (val, idx) in enumerate(zip(tok_vals.tolist(), tok_idx.tolist()), 1):
        print(f"  {r:2d}. id={idx:5d}  sim={val: .4f}  token='{id_to_token(idx)}'")

    #print(f"\nClosest positions to token_id={token_id} ({id_to_token(token_id)}):")
    #for r, (val, idx) in enumerate(zip(pos_vals.tolist(), pos_idx.tolist()), 1):
    #    print(f"  {r:2d}. pos={idx:4d}  sim={val: .4f}")

# ---- 2) Neighbors of a position id among POSITIONS and TOKENS ----
def nearest_to_position(pos_id, k_positions=10, k_tokens=10):
    q = pos_w[pos_id]  # [D]
    pos_vals, pos_idx = cosine_topk(pos_w, q, k=k_positions, exclude_idx=pos_id)
    tok_vals, tok_idx = cosine_topk(tok_w, q, k=k_tokens, exclude_idx=None)

    print(f"Closest positions to pos_id={pos_id}:")
    for r, (val, idx) in enumerate(zip(pos_vals.tolist(), pos_idx.tolist()), 1):
        print(f"  {r:2d}. pos={idx:4d}  sim={val: .4f}")

    #print(f"\nClosest tokens to pos_id={pos_id}:")
    #for r, (val, idx) in enumerate(zip(tok_vals.tolist(), tok_idx.tolist()), 1):
    #    print(f"  {r:2d}. id={idx:5d}  sim={val: .4f}  token='{id_to_token(idx)}'")

# ---- 3) Neighbors to ANY custom vector (e.g., a hidden state projected to embed dim) ----
def nearest_to_vector(vec, search="tokens", k=10):
    """
    vec: [D] torch tensor in the same embedding space
    search: "tokens", "positions", or "both"
    """
    vec = vec.to(device)
    if search in ("tokens", "both"):
        tok_vals, tok_idx = cosine_topk(tok_w, vec, k=k)
        print("Closest TOKENS:")
        for r, (val, idx) in enumerate(zip(tok_vals.tolist(), tok_idx.tolist()), 1):
            print(f"  {r:2d}. id={idx:5d}  sim={val: .4f}  token='{id_to_token(idx)}'")

    if search in ("positions", "both"):
        pos_vals, pos_idx = cosine_topk(pos_w, vec, k=k)
        print("\nClosest POSITIONS:")
        for r, (val, idx) in enumerate(zip(pos_vals.tolist(), pos_idx.tolist()), 1):
            print(f"  {r:2d}. pos={idx:4d}  sim={val: .4f}")

# ===== Example usage =====
# nearest_to_token(token_id=123, k_tokens=15, k_positions=8)
# nearest_to_position(pos_id=10, k_positions=15, k_tokens=8)
# some_vec = torch.randn(tok_w.shape[1], device=device)  # example custom query
# nearest_to_vector(some_vec, search="both", k=10)


In [ ]:
stoi

In [ ]:
# a = 10
# e = 14
nearest_to_token(token_id=17, k_tokens=15, k_positions=8)

In [ ]:
nearest_to_position(pos_id=0, k_positions=15, k_tokens=8)

In [ ]:
print(model.token_embedding_table)    # token from the vocabulary
print(model.position_embedding_table) # position in the sentence
print()

# The weight matrix shape is (num_tokens, embedding_dim)
embedding_weight = model.token_embedding_table.weight
# Convert to NumPy array:
embedding_matrix = embedding_weight.detach().cpu().numpy()
print(type(embedding_matrix))   # <class 'numpy.ndarray'>
print(embedding_matrix.shape)   # (vocab_size, embedding_dim)
print(embedding_matrix[1])
print()

# Extract both values:
vocab_size, embedding_dim = embedding_weight.shape

print(f"Vocabulary size: {vocab_size}")
print(f"Embedding dimensionality: {embedding_dim}")
print()

# The weight matrix shape is (num_positions, embedding_dim2)
position_embedding_weight = model.position_embedding_table.weight
position_embedding_matrix = position_embedding_weight.detach().cpu().numpy()
print(type(position_embedding_matrix))   # <class 'numpy.ndarray'>
print(position_embedding_matrix.shape)   # (vocab_size, embedding_dim)
print(position_embedding_matrix[1])
print()

# Extract both values:
positions_size, position_embedding_dim = position_embedding_weight.shape

print(f"Positions size: {positions_size}")
print(f"Position embedding dimensionality: {position_embedding_dim}")
print()

# available: embedding_matrix, position_embedding_matrix
#            vocab_size, embedding_dim
#            positions_size, position_embedding_dim
for i in range(len(itos)):
    print(i, myitos[i], embedding_matrix[i])
    print(i, myitos[i])

In [ ]:
import numpy as np

# ---------------------------
# 1) Distances between vectors
# ---------------------------

def cosine_distance(u: np.ndarray, v: np.ndarray, eps: float = 1e-12) -> float:
    """
    Cosine distance = 1 - cosine similarity.
    """
    u = np.asarray(u, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    num = np.dot(u, v)
    den = (np.linalg.norm(u) * np.linalg.norm(v)) + eps
    return 1.0 - (num / den)

def euclidean_distance(u: np.ndarray, v: np.ndarray) -> float:
    """
    Standard L2 distance.
    """
    u = np.asarray(u, dtype=np.float64)
    v = np.asarray(v, dtype=np.float64)
    return float(np.linalg.norm(u - v))

# Example (two 32-d vectors):
# u = np.random.randn(32)
# v = np.random.randn(32)
# print("Cosine distance:", cosine_distance(u, v))
# print("Euclidean distance:", euclidean_distance(u, v))


# -----------------------------------------
# 2) Find the 7 closest vectors to a target
# -----------------------------------------

def _pairwise_distances_to_x(X: np.ndarray, x: np.ndarray, metric: str = "cosine") -> np.ndarray:
    """
    Compute distances from every row in X to vector x.
    Supports 'cosine' (default) and 'euclidean'.
    """
    X = np.asarray(X, dtype=np.float64)
    x = np.asarray(x, dtype=np.float64)

    if metric == "cosine":
        # 1 - (X·x) / (||X|| * ||x||)
        x_norm = np.linalg.norm(x) + 1e-12
        X_norms = np.linalg.norm(X, axis=1) + 1e-12
        dots = X @ x
        return 1.0 - (dots / (X_norms * x_norm))

    elif metric == "euclidean":
        # ||X - x||_2 for each row
        diffs = X - x
        return np.sqrt(np.einsum("ij,ij->i", diffs, diffs))

    else:
        raise ValueError("metric must be 'cosine' or 'euclidean'")

def top_k_nearest(
    X: np.ndarray,
    x: np.ndarray,
    ascending: bool = True,
    k: int = 7,
    metric: str = "cosine",
    exclude_self_if_present: bool = True
):
    """
    Return indices and distances of the k nearest rows in X to x
    using the chosen metric. If x is exactly present in X (within
    atol/rtol), it will be excluded when exclude_self_if_present=True.
    """
    X = np.asarray(X, dtype=np.float64)
    x = np.asarray(x, dtype=np.float64)

    if X.ndim != 2:
        raise ValueError("X must be a 2D array of shape (n_vectors, dim)")
    if x.ndim != 1 or x.shape[0] != X.shape[1]:
        raise ValueError(f"x must be shape ({X.shape[1]},), got {x.shape}")

    dists = _pairwise_distances_to_x(X, x, metric=metric)

    # Optionally exclude the exact self vector if present
    if exclude_self_if_present:
        # Identify rows that match x within tolerance
        same = np.isclose(X, x, rtol=1e-6, atol=1e-8).all(axis=1)
        dists = np.where(same, np.inf, dists)

    # Handle k larger than available candidates
    k_eff = min(k, (X.shape[0] - int(exclude_self_if_present)))

    # Use argpartition for O(n) selection, then fully sort that slice
    if ascending:
        idx_k = np.argpartition(dists, k_eff)[:k_eff]
    else:
        idx_k = np.argpartition(dists, k_eff)[k_eff:]
    idx_sorted = idx_k[np.argsort(dists[idx_k])]
    return idx_sorted, dists[idx_sorted]

# -----------------
# Example end-to-end
# -----------------
if __name__ == "__main__":
    # Suppose we have 42 embedding vectors of size 32:
    # rng = np.random.default_rng(7)
    # X = rng.normal(size=(42, 64))
    
    # available: embedding_matrix, position_embedding_matrix
    #            vocab_size, embedding_dim
    #            positions_size, position_embedding_dim

    X = embedding_matrix

    # Choose a target x (can be one from X or any 64-d vector)
    selected = 21 # a:17 e:21 ž:41 
    x = X[selected]  # e.g., the Nth vector in the set

    # Get the 7 closest (excluding the self-match)
    indices, distances = top_k_nearest(X, x, ascending = True, k=vocab_size, metric="cosine")

    print(f"nearest to farthest neighbors to {myitos[selected]} (by cosine distance):")
    for rank, (i, d) in enumerate(zip(indices, distances), start=1):
        print(f"{rank:>2}. index={i:>2}  {myitos[i]} distance={d:.6f}")

    # If you prefer Euclidean:
    # indices_eu, distances_eu = top_k_nearest(X, x, k=7, metric="euclidean")
